# Train diffusion — GPU slice A

**Kernel:** `conda env:.conda-diffusion` (one GPU / MIG slice per notebook)

**Queue (sequential):** `linear` → `ramp` → `pred_xstart`

Run this notebook on one slice; run the companion notebook on the other.
Keep `SEQUENTIAL = True`.

Flag bundles: `configs/train_flags/{name}.sh`  
Defaults match Gray **iterE** plus wall-time fixes (validation_interval 500, fp16, max_steps 111000).  
Set `DRY_RUN = False` before launching.

With `RESUME = True` (default), each config auto-resumes from the latest
`brats2updateNNNNNN.pt` in its log dir and continues until `MAX_STEPS`.
`progress.csv` is seeded from durable storage (if scratch was wiped) and the
logger **appends** on resume so the learning curve is not truncated.

Scratch / data dirs are **discovered at runtime** from existing `/scratch/7Day*` pools
(never hardcoded). Re-run all cells from the top after a kernel reconnect.


In [ ]:
from __future__ import annotations

import importlib
import json
import os
import shlex
import subprocess
import sys
import time
from pathlib import Path

APP_ROOT = Path("/exp/sbnd/app/users/munjung/anomaly-detection")
sys.path.insert(0, str(APP_ROOT))

# Drop stale configs.paths if this kernel imported an older copy.
if "configs.paths" in sys.modules:
    importlib.reload(sys.modules["configs.paths"])
if "configs" in sys.modules:
    importlib.reload(sys.modules["configs"])

from configs.paths import (
    DATA_ROOT,
    SCRATCH_ROOT,
    describe_environment,
    ensure_layout,
    existing_scratch_pools,
    require_existing_dir,
    resolve_training_data_dir,
    resolve_training_run_root,
    seed_progress_csv,
)
from configs.train_configs import TRAIN_CONFIGS, list_configs_table, resolve_flag_script

ensure_layout()
print(describe_environment())
print()
print(list_configs_table())

discover = {}
for p in [DATA_ROOT / "training" / "eaf_discover.json", APP_ROOT / "train" / "eaf_discover.json"]:
    if p.is_file():
        discover = json.loads(p.read_text())
        print("loaded", p)
        break

# Prefer EAF-discovered checkout if it has anisotropic support; else local synced tree.
_candidates = []
if discover.get("diffusion_root"):
    _candidates.append(Path(discover["diffusion_root"]))
_candidates.append(APP_ROOT / "train" / "diffusion-anomaly")

DIFFUSION_ROOT = None
for c in _candidates:
    if (c / "scripts" / "image_train.py").exists() and (
        c / "guided_diffusion" / "anisotropic_diffusion.py"
    ).exists():
        DIFFUSION_ROOT = c
        break
if DIFFUSION_ROOT is None:
    DIFFUSION_ROOT = _candidates[-1]
    print("WARNING: anisotropic_diffusion.py missing — anisotropic config will fail")

assert (DIFFUSION_ROOT / "scripts" / "image_train.py").exists(), DIFFUSION_ROOT
print("DIFFUSION_ROOT =", DIFFUSION_ROOT)


In [ ]:
# --- Run controls (slice A) ---
DRY_RUN = False          # False → actually launch training
SEQUENTIAL = True       # keep True on a single GPU slice
RESUME = True           # load latest brats2update*.pt and continue
CONFIGS_TO_RUN = ['linear', 'ramp', 'pred_xstart']

CHARGE_SCALE = "1"  # iterE flags omitted this (default 1); weight_pixels=False
MAX_STEPS = "111000"  # ~iterE production ckpt; constant LR (lr_anneal_steps=0)

# Resolve paths from what actually exists on this host (no hardcoded 7Day* name).
print("scratch pools on this host:", [str(p) for p in existing_scratch_pools()] or ["<none>"])
print("discovered SCRATCH_ROOT:", SCRATCH_ROOT)

DATA_DIR = resolve_training_data_dir(discover)
VAL_DIR = DATA_DIR  # same as train (matches Gray iterE); set a held-out dir if desired
RUN_ROOT = resolve_training_run_root()
LEGACY_RESULTS = APP_ROOT / "train" / "diffusion-anomaly" / "results"

print("SLICE  ", "A")
print("DATA_DIR", DATA_DIR, "exists=", DATA_DIR.exists())
print("RUN_ROOT", RUN_ROOT, "exists=", RUN_ROOT.exists())
print("CONFIGS", CONFIGS_TO_RUN)
print("DRY_RUN", DRY_RUN, "RESUME", RESUME)
print("MAX_STEPS", MAX_STEPS, "CHARGE_SCALE", CHARGE_SCALE)

require_existing_dir(DATA_DIR, "DATA_DIR")
require_existing_dir(RUN_ROOT, "RUN_ROOT")


In [ ]:
import re


def checkpoint_step(path: Path) -> int:
    m = re.search(r"brats2update(\d+)\.pt$", path.name)
    return int(m.group(1)) if m else -1


def find_latest_checkpoint(config_name: str, log_dir: Path) -> Path | None:
    """Return newest brats2update*.pt for this config, or None."""
    search_dirs = [
        log_dir,
        DATA_ROOT / "training" / "diffusion" / config_name,
    ]
    # Pre-OPENAI_LOGDIR-fix runs shared a single results/ folder.
    legacy = globals().get("LEGACY_RESULTS", APP_ROOT / "train" / "diffusion-anomaly" / "results")
    if config_name in ("linear", "anisotropic") and Path(legacy).is_dir():
        search_dirs.append(Path(legacy))

    best: Path | None = None
    best_step = -1
    for d in search_dirs:
        if not d.is_dir():
            continue
        for ckpt in d.glob("brats2update*.pt"):
            step = checkpoint_step(ckpt)
            if step > best_step:
                best_step = step
                best = ckpt
    return best


def build_train_command(config_name: str) -> tuple[Path, Path, str, Path | None]:
    flag = resolve_flag_script(config_name)
    assert flag.is_file(), flag
    log_dir = RUN_ROOT / config_name
    log_dir.mkdir(parents=True, exist_ok=True)
    log_file = log_dir / "train.log"
    # Scratch may be empty after pool expiry — restore the longest durable curve first.
    seeded = seed_progress_csv(log_dir, config_name)
    if seeded is not None:
        print(f" progress.csv ready ({sum(1 for _ in seeded.open()) - 1} rows) → {seeded}")

    do_resume = globals().get("RESUME", True)
    resume_ckpt = find_latest_checkpoint(config_name, log_dir) if do_resume else None
    resume_ckpt_str = str(resume_ckpt) if resume_ckpt is not None else ""

    # Pass resume path as its own quoted argv (avoid nested-quote bugs in IMAGE_TRAIN_FLAGS).
    cmd = f"""
set -euo pipefail
cd "{DIFFUSION_ROOT}"
export PYTHONPATH="/exp/sbnd/app/users/munjung/anomaly-detection/_stubs:{DIFFUSION_ROOT}:$PYTHONPATH"
export PYTORCH_ALLOC_CONF="${{PYTORCH_ALLOC_CONF:-expandable_segments:False}}"
unset PYTORCH_CUDA_ALLOC_CONF
# Do not overwrite CUDA_VISIBLE_DEVICES — Jupyter/MIG already set it.
echo "[$(date -Is)] CUDA_VISIBLE_DEVICES=${{CUDA_VISIBLE_DEVICES:-<unset>}}"
echo "[$(date -Is)] PYTORCH_ALLOC_CONF=$PYTORCH_ALLOC_CONF"
export DATA_DIR="{DATA_DIR}"
export VAL_DIR="{VAL_DIR}"
export CHARGE_SCALE="{CHARGE_SCALE}"
source "{flag}"
export IMAGE_TRAIN_FLAGS="$IMAGE_TRAIN_FLAGS --max_steps {MAX_STEPS}"
export OPENAI_LOGDIR="{log_dir}"
mkdir -p "$OPENAI_LOGDIR"
RESUME_CKPT="{resume_ckpt_str}"
echo "[$(date -Is)] CONFIG=$CONFIG_NAME"
echo "[$(date -Is)] FLAG={flag}"
echo "[$(date -Is)] OPENAI_LOGDIR=$OPENAI_LOGDIR"
echo "[$(date -Is)] DATA_DIR=$DATA_DIR"
echo "[$(date -Is)] RESUME_CHECKPOINT=${{RESUME_CKPT:-<none>}}"
echo "[$(date -Is)] IMAGE_TRAIN_FLAGS=$IMAGE_TRAIN_FLAGS"
if [ -n "$RESUME_CKPT" ]; then
  python scripts/image_train.py $IMAGE_TRAIN_FLAGS --resume_checkpoint "$RESUME_CKPT" 2>&1 | tee -a "{log_file}"
else
  python scripts/image_train.py $IMAGE_TRAIN_FLAGS 2>&1 | tee -a "{log_file}"
fi
"""
    return flag, log_dir, cmd, resume_ckpt


jobs = {}
for name in CONFIGS_TO_RUN:
    flag, log_dir, cmd, resume_ckpt = build_train_command(name)
    jobs[name] = {"flag": flag, "log_dir": log_dir, "cmd": cmd, "resume": resume_ckpt}
    print(f"\n=== {name} ===")
    print(" flag   ", flag)
    print(" log_dir", log_dir)
    if resume_ckpt is not None:
        print(f" RESUME  step {checkpoint_step(resume_ckpt)} ← {resume_ckpt}")
    else:
        print(" RESUME  <none> (fresh start)" if globals().get("RESUME", True) else " RESUME  disabled")


In [ ]:
# Preview / launch
# Re-run cells above after a kernel reconnect so jobs / RESUME / DIFFUSION_ROOT exist.
missing = [n for n in ("jobs", "DIFFUSION_ROOT", "DRY_RUN", "SEQUENTIAL") if n not in globals()]
if missing:
    raise RuntimeError(
        f"Missing {missing}. Re-run all cells from the top after a session reload, "
        "then run this launch cell."
    )

procs = {}
for name, job in jobs.items():
    print("\n" + "=" * 60)
    print(f"CONFIG {name}")
    if job.get("resume") is not None:
        print(f"will resume from {job['resume']}")
    else:
        print("fresh start (no checkpoint found)" if globals().get("RESUME", True) else "resume disabled")
    if DRY_RUN:
        print("(dry-run) would run:")
        print(job["cmd"])
        continue

    wrapper = job["log_dir"] / "launch.sh"
    wrapper.write_text(job["cmd"])
    wrapper.chmod(0o755)
    log_file = job["log_dir"] / "train.log"
    if SEQUENTIAL:
        print("launching sequential:", name)
        ret = subprocess.run(["bash", str(wrapper)], cwd=str(DIFFUSION_ROOT))
        print(name, "exit", ret.returncode)
        if ret.returncode != 0:
            if log_file.is_file():
                print("--- last 40 log lines ---")
                print("\n".join(log_file.read_text(errors="replace").splitlines()[-40:]))
            raise SystemExit(f"{name} failed with {ret.returncode}")
    else:
        print("launching background:", name)
        procs[name] = subprocess.Popen(["bash", str(wrapper)], cwd=str(DIFFUSION_ROOT))

if not DRY_RUN and not SEQUENTIAL and procs:
    print("waiting on", list(procs))
    for name, p in procs.items():
        rc = p.wait()
        print(name, "exit", rc)


## Monitor (slice A)

Log dirs are under the discovered `RUN_ROOT` printed above.

```python
for name in ['linear', 'ramp', 'pred_xstart']:
    print(RUN_ROOT / name / 'train.log')
```

```bash
tail -f $RUN_ROOT/<config>/train.log
```

On relaunch, look for `RESUME_CHECKPOINT=.../brats2updateNNNNNN.pt` and
`loading model from checkpoint` / `loading optimizer state` in the log.

Learning curves: `train/03_LearningCurves.ipynb`.
